# Subscription Information Extraction with DistilBERT

**Dol Raj Bashyal**  
**Task:** Named entity recognition (NER) for natural-language subscription entry

## Research overview

Entering subscription information manually requires users to identify several details in a single description. This project investigates whether a fine-tuned, pretrained language model can recognize eight subscription fields in free-form text. It uses a labeled dataset of synthetic and separately authored examples, trains a DistilBERT token classifier, and evaluates both individual entity extraction and complete-description accuracy.

**Objectives:** Prepare BIO-labeled training data; fine-tune a lightweight transformer; select the best checkpoint and confidence threshold using development data; and evaluate extraction performance on a separately authored final set. Finally use the tuned model for personal project in an ios application.

**Scope:** This experiment concerns extracting text spans, not resolving ambiguous dates into calendar timestamps or creating actual subscription records.

## 1. Task definition

The model identifies eight entity types: **service, amount, currency, billing cycle, billing date, trial end, reminder, and plan type**. A single description may contain several fields, and unrelated text may contain none.

**Example input:** “Netflix costs 15.99 USD per month, renews October 25, and remind me two days before.”

**Approach:** Token classification assigns one of 17 BIO labels to each token: `O`, plus `B-` and `I-` for each of the eight entity types. Predicted token labels are merged into text spans.

**Primary outcome:** Exact-match entity precision, recall and F1. **Additional outcome:** Complete-entry accuracy, which requires all entities in a description to match.

In [ ]:
# Install compatible Hugging Face dependencies only when needed.

import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from packaging.specifiers import SpecifierSet

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

requirements = {
    'transformers': '==5.10.1',
    'accelerate': '>=1.12.0,<2',
    'huggingface-hub': '>=1.23.0,<2',
}
needs_install = False
for package, constraint in requirements.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed is None or installed not in SpecifierSet(constraint):
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ('transformers', 'accelerate', 'huggingface_hub'))
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
        *[name + constraint for name, constraint in requirements.items()],
    ], check=True)
    if loaded:
        raise RuntimeError('Packages updated. Restart the Kaggle session, then run all cells again.')

In [ ]:
# Import research libraries and configure dataset and output paths.

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from IPython.display import display, Markdown
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, Trainer, TrainingArguments, set_seed,
)

MODEL_ID = 'distilbert/distilbert-base-cased'
MODEL_REVISION = '6ea81172465e8b0ad3fddeed32b986cdcdcffcf0'
MAX_TOKENS = 128
SEED = 326
FIELDS = ('SERVICE', 'AMOUNT', 'CURRENCY', 'CYCLE', 'BILLING_DATE', 'TRIAL_END', 'REMINDER', 'PLAN_TYPE')
LABELS = ['O'] + [f'{prefix}-{field}' for field in FIELDS for prefix in ('B', 'I')]
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
ID_TO_LABEL = dict(enumerate(LABELS))
FIELD_NAMES = dict(zip(FIELDS, [
    'Service', 'Amount', 'Currency', 'Billing cycle',
    'Billing date', 'Trial end', 'Reminder', 'Plan type',
]))
set_seed(SEED)
torch.set_num_threads(4)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlepad': 16, 'axes.labelcolor': '#333945', 'text.color': '#333945',
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'figure.dpi': 110, 'savefig.dpi': 160,
})
pd.set_option('display.max_colwidth', 120)


def read_json(path):
    """Read a UTF-8 JSON file from a pathlib path."""
    return json.loads(Path(path).read_text())


input_override = os.environ.get('DUE_INPUT_DIR')
if input_override:
    INPUT_DIR = Path(input_override)
else:
    search_root = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
    matches = []
    for manifest_path in search_root.rglob('manifest.json'):
        try:
            if read_json(manifest_path).get('dataset') == 'Due subscription information extraction':
                matches.append(manifest_path.parent)
        except (OSError, ValueError):
            continue
    if len(matches) != 1:
        raise FileNotFoundError('Attach one Due subscription dataset with Add Input, or set DUE_INPUT_DIR to its extracted folder.')
    INPUT_DIR = matches[0]

working = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'output'
run_name = 'due-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUTPUT_DIR = Path(os.environ.get('DUE_OUTPUT_DIR', working / run_name))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
FIGURES = OUTPUT_DIR / 'figures'
FIGURES.mkdir()


def show_figure(figure, name):
    """Save a plot under the current run and display it in the notebook."""
    figure.savefig(FIGURES / f'{name}.png', bbox_inches='tight')
    plt.show()
    plt.close(figure)


manifest = read_json(INPUT_DIR / 'manifest.json')
print('Dataset:', manifest['dataset'])
display(pd.DataFrame({
    'Package': ['torch', 'transformers', 'accelerate', 'huggingface-hub'],
    'Version': [version(package) for package in ['torch', 'transformers', 'accelerate', 'huggingface-hub']],
}))

## 2. Dataset and provenance

The training data was created from **58 subscription patterns** and **20 unrelated-text patterns**, with varying names, prices, currencies, dates, and reminders. Duplicates were removed and descriptions were shuffled using seed **32602**. Development and final examples were authored separately. These are hypothetical descriptions, not collected customer data.

The raw CSV contains **3,796 training**, **62 development**, and **84 final** descriptions. `annotated_text` uses tags such as `[[SERVICE|Netflix]]` to identify the expected fields.

**Data limitation:** The training set is generated from templates rather than collected user entries. The separately authored development and final sets test generalization beyond those generated sentences, but results may not reflect real-world subscription language.

In [ ]:
# Load the original annotated descriptions and validate dataset structure.

raw = pd.read_csv(INPUT_DIR / 'raw/subscription_descriptions.csv', keep_default_na=False)
assert raw['id'].is_unique
assert set(raw['split']) == {'train', 'development', 'final'}
assert raw['text'].str.strip().ne('').all()
print('Descriptions:', len(raw))
display(raw.head(4))
patterns = read_json(INPUT_DIR / 'raw/sentence_patterns.json')
print('Example generation pattern:')
print(patterns['paid_patterns'][0])

## 3. Annotation processing and quality checks

Annotation tags are removed to create the input sentence. Their values become entity labels and character offsets. Each split is reconstructed from the raw CSV and checked against the original dataset fingerprints.

In [ ]:
# Convert inline [[FIELD|value]] tags into span annotations, then verify fingerprints.

def parse_annotations(source, origin, family):
    """Remove inline tags and record labeled character-span offsets."""
    text = ''
    entities = []
    cursor = 0
    for match in re.finditer(r'\[\[([A-Z_]+)\|(.*?)\]\]', source):
        text += source[cursor:match.start()]
        label, value = match.groups()
        if label not in FIELDS:
            raise ValueError(f'Unknown field: {label}')
        start = len(text)
        text += value
        entities.append({'label': label, 'start': start, 'end': len(text), 'text': value})
        cursor = match.end()
    text += source[cursor:]
    return {'text': text, 'entities': entities, 'origin': origin, 'family': family}


processed_dir = OUTPUT_DIR / 'processed'
processed_dir.mkdir()
records = {split: [] for split in ('train', 'development', 'final')}
for row in raw.itertuples(index=False):
    record = parse_annotations(row.annotated_text, row.origin, row.template_family)
    assert record['text'] == row.text, f'Annotation/text mismatch: {row.id}'
    records[row.split].append(record)
for split, descriptions in records.items():
    destination = processed_dir / f'{split}.jsonl'
    destination.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in descriptions))
    fingerprint = hashlib.sha256(destination.read_bytes()).hexdigest()
    assert fingerprint == manifest['splits'][split]['sha256'], split

sample = records['train'][0]
print(sample['text'])
display(pd.DataFrame(sample['entities']))

In [ ]:
# Validate span offsets and confirm the train/development/final splits do not overlap.

def read_records(path):
    """Load JSONL examples and verify unique text and valid offsets."""
    records = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    seen = set()
    for record in records:
        text = record['text']
        if not text.strip() or text in seen:
            raise ValueError('Empty or duplicate input in dataset')
        seen.add(text)
        previous_end = 0
        for entity in record['entities']:
            start, end, label = entity['start'], entity['end'], entity['label']
            if label not in FIELDS or not (previous_end <= start < end <= len(text)):
                raise ValueError(f'Invalid entity: {entity}')
            if text[start:end] != entity['text']:
                raise ValueError('Entity text does not match its character offsets')
            previous_end = end
    return records


records = {split: read_records(processed_dir / f'{split}.jsonl') for split in records}
summary = []
for split, descriptions in records.items():
    positive = sum(bool(row['entities']) for row in descriptions)
    summary.append({
        'Split': split, 'Descriptions': len(descriptions),
        'Subscription descriptions': positive,
        'Unrelated descriptions': len(descriptions) - positive,
        'Entities': sum(len(row['entities']) for row in descriptions),
    })
display(pd.DataFrame(summary).set_index('Split'))

texts = {split: {row['text'] for row in descriptions} for split, descriptions in records.items()}
for first, second in [('train', 'development'), ('train', 'final'), ('development', 'final')]:
    assert texts[first].isdisjoint(texts[second]), f'Duplicate description across {first} and {second}'

### Field distribution

In [ ]:
# Count how many annotated examples of each field appear in each split.

field_counts = pd.DataFrame({
    split: Counter(entity['label'] for row in descriptions for entity in row['entities'])
    for split, descriptions in records.items()
}).reindex(FIELDS).fillna(0).astype(int)
field_counts.index = [FIELD_NAMES[field] for field in FIELDS]
display(field_counts)
fig, ax = plt.subplots(figsize=(10, 4.5))
field_counts['train'].plot.bar(ax=ax, color='#526AA2', width=0.65)
ax.set(title='Training entities by field', xlabel='', ylabel='Entity spans')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.15)
show_figure(fig, 'training-field-distribution')

### Tokenization and BIO labels

Cased WordPiece tokenization preserves capitalization. `B` begins an entity, `I` continues it, and `O` marks other words. Eight fields produce **17 token labels**. Special tokens use the ignored training label `-100`. Inputs are limited to **128 tokens**.

In [ ]:
# Align character-level entity spans with DistilBERT WordPiece tokens using BIO labels.

def encode_record(record, tokenizer):
    """Tokenize one record and align entities with BIO label IDs."""
    encoded = tokenizer(record['text'], return_offsets_mapping=True, truncation=False)
    if len(encoded['input_ids']) > MAX_TOKENS:
        raise ValueError('Input exceeds 128 tokens; shorten it rather than silently truncating')
    labels, previous_entity = [], None
    covered = set()
    for start, end in encoded.pop('offset_mapping'):
        if start == end:
            labels.append(-100)
            previous_entity = None
            continue
        match = None
        for i, entity in enumerate(record['entities']):
            if start < entity['end'] and end > entity['start']:
                if start < entity['start'] or end > entity['end']:
                    raise ValueError(f'Annotation cuts through a token: {record["text"][start:end]}')
                match = i
                break
        if match is None:
            labels.append(LABEL_TO_ID['O'])
        else:
            field = record['entities'][match]['label']
            labels.append(LABEL_TO_ID[f'{"I" if previous_entity == match else "B"}-{field}'])
            covered.add(match)
        previous_entity = match
    if len(covered) != len(record['entities']):
        raise ValueError('An annotated entity was not covered by any tokens')
    encoded['labels'] = labels
    return encoded


tokenizer = AutoTokenizer.from_pretrained(INPUT_DIR / 'tokenizer', use_fast=True, local_files_only=True)
encoded_splits = {split: [encode_record(row, tokenizer) for row in descriptions] for split, descriptions in records.items()}
sample = records['development'][0]
encoded = encode_record(sample, tokenizer)
offsets = tokenizer(sample['text'], return_offsets_mapping=True)['offset_mapping']
print(sample['text'])
display(pd.DataFrame([
    {'Token': token, 'Start': start, 'End': end, 'BIO label': 'Ignored' if label == -100 else LABELS[label]}
    for token, (start, end), label in zip(
        tokenizer.convert_ids_to_tokens(encoded['input_ids']), offsets, encoded['labels']
    )
]))

In [ ]:
# Inspect token-length distributions to check the 128-token input limit.

lengths = {split: [len(row['input_ids']) for row in descriptions] for split, descriptions in encoded_splits.items()}
display(pd.DataFrame({
    split: {'Minimum': min(values), 'Median': np.median(values), 'Maximum': max(values)}
    for split, values in lengths.items()
}).T)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (split, values), color in zip(axes, lengths.items(), ['#8C98AD', '#526AA2', '#B497AA']):
    ax.hist(values, bins=np.arange(0, 130, 5), color=color, edgecolor='white')
    ax.axvline(MAX_TOKENS, color='#AC785D', linestyle='--')
    ax.set(title=split.capitalize(), xlabel='WordPiece tokens', ylabel='Descriptions', xlim=(0, 132))
fig.tight_layout()
show_figure(fig, 'input-lengths')

## 4. Experimental split and evaluation protocol

The three sets have distinct roles: **training (3,796)** updates model parameters, **development (62)** selects a checkpoint and confidence threshold, and **final (84)** evaluates the selected procedure. The notebook checks that descriptions do not overlap across these sets.

The final examples were evaluated in the original experiment and are therefore an **existing benchmark**, not a newly untouched external test set. Reported baseline predictions are retained for context and are clearly distinguished from models trained in this run.

## 5. Model and design rationale

The model is [DistilBERT base cased](https://huggingface.co/distilbert/distilbert-base-cased), a pretrained transformer with a task-specific 17-label token-classification head. Its **cased** tokenizer preserves distinctions in names such as services and plans.

**Why DistilBERT?** It offers a smaller, more computationally efficient alternative to full BERT while retaining contextual language representations. Fine-tuning allows the model to adapt those representations to the eight subscription fields. This study tests whether that trade-off is suitable for the extraction task; it does **not** benchmark model size, speed, or device inference time in this notebook.

The model predicts character-aligned entity spans using BIO token labels; these spans can be inspected or mapped to structured fields by a downstream application.

In [ ]:
# Decode BIO labels into entity spans and calculate exact-match entity metrics.

def decode_entities(text, offsets, label_ids, probabilities=None):
    """Reconstruct character spans from predicted BIO token labels."""
    entities, current = [], None
    for i, ((start, end), label_id) in enumerate(zip(offsets, label_ids)):
        label = ID_TO_LABEL[int(label_id)]
        if start == end or label == 'O':
            if current:
                entities.append(current)
                current = None
            continue
        prefix, field = label.split('-', 1)
        confidence = float(probabilities[i]) if probabilities is not None else 1.0
        if prefix == 'I' and current and current['label'] == field:
            current['end'] = end
            current['confidence'] = min(current['confidence'], confidence)
        else:
            if current:
                entities.append(current)
            current = {'label': field, 'start': start, 'end': end, 'confidence': confidence}
    if current:
        entities.append(current)
    for entity in entities:
        entity['text'] = text[entity['start']:entity['end']]
    return entities

def score_records(records, predictions):
    """Compute exact-boundary entity scores and full-description accuracy."""
    if len(records) != len(predictions):
        raise ValueError('Every test record needs exactly one prediction')
    true_positive = false_positive = false_negative = exact = 0
    by_field = {field: {'tp': 0, 'fp': 0, 'fn': 0} for field in FIELDS}
    for record, predicted in zip(records, predictions):
        gold = {(e['label'], e['start'], e['end']) for e in record['entities']}
        found = {(e['label'], e['start'], e['end']) for e in predicted}
        exact += gold == found
        true_positive += len(gold & found)
        false_positive += len(found - gold)
        false_negative += len(gold - found)
        for field in FIELDS:
            g = {e for e in gold if e[0] == field}
            p = {e for e in found if e[0] == field}
            by_field[field]['tp'] += len(g & p)
            by_field[field]['fp'] += len(p - g)
            by_field[field]['fn'] += len(g - p)
    precision = true_positive / max(1, true_positive + false_positive)
    recall = true_positive / max(1, true_positive + false_negative)
    return {
        'entity_precision': precision, 'entity_recall': recall,
        'entity_f1': 2 * precision * recall / max(1e-12, precision + recall),
        'exact_extraction_rate': exact / max(1, len(records)),
        'records': len(records), 'by_field_counts': by_field,
        'note': 'Extraction accuracy, not normalized draft accuracy or a user-study result.',
    }

In [ ]:
# Calculate token accuracy and span-level metrics for checkpoint selection.

def token_metrics(result):
    """Compute accuracy over tokens that have valid reference labels."""
    predictions, labels = result
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predicted = np.argmax(predictions, axis=-1)
    valid = labels != -100
    entities = valid & (labels != 0)
    return {
        'token_accuracy': float((predicted[valid] == labels[valid]).mean()),
        'entity_token_accuracy': float((predicted[entities] == labels[entities]).mean()) if entities.any() else 0.0,
    }

def span_metrics(records, tokenizer):
    """Return a Trainer metric function combining token and entity scores."""
    offsets = [tokenizer(row['text'], return_offsets_mapping=True)['offset_mapping'] for row in records]
    def compute(result):
        metrics = token_metrics(result)
        logits = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
        labels = logits.argmax(-1)
        predictions = [decode_entities(row['text'], offset, tags[:len(offset)])
                       for row, offset, tags in zip(records, offsets, labels)]
        scores = score_records(records, predictions)
        metrics.update({key:scores[key] for key in ('entity_f1','entity_precision','entity_recall','exact_extraction_rate')})
        return metrics
    return compute

## 6. Fine-tuning procedure

Fine-tune all model layers and a new token-classification head for **four epochs**, with training batch size **16**, learning rate **3 × 10⁻⁵**, weight decay **0.01**, and random seed **326**. The best checkpoint is selected using development **entity F1** rather than final-set performance.

The following cell defines the training dataset, sets the hyperparameters, runs the Hugging Face `Trainer`, and saves the selected weights plus training metadata.

In [ ]:
# Fine-tune DistilBERT on training data; choose the best checkpoint on development F1.

class TokenDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION,
    id2label=ID_TO_LABEL, label2id=LABEL_TO_ID, attn_implementation='eager',
)
arguments = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    learning_rate=3e-5, weight_decay=0.01,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=4, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='entity_f1',
    greater_is_better=True, save_total_limit=2,
    logging_steps=20, report_to='none', seed=SEED, data_seed=SEED,
    dataloader_num_workers=0,
)
trainer = Trainer(
    model=model, args=arguments,
    train_dataset=TokenDataset(encoded_splits['train']),
    eval_dataset=TokenDataset(encoded_splits['development']),
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=span_metrics(records['development'], tokenizer),
)
training_result = trainer.train()
model = trainer.model.eval()
MODEL_DIR = OUTPUT_DIR / 'model'
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(MODEL_DIR)
trainer.save_state()
run_metadata = {
    'base_model': MODEL_ID, 'base_revision': MODEL_REVISION, 'seed': SEED,
    'completed_epochs': trainer.state.epoch, 'completed_steps': trainer.state.global_step,
    'best_checkpoint_step': trainer.state.best_global_step,
    'training_metrics': training_result.metrics, 'device': str(trainer.args.device),
    'training_examples': len(records['train']), 'label_order': LABELS,
    'training_sha256': manifest['splits']['train']['sha256'],
}
(MODEL_DIR / 'training_metadata.json').write_text(json.dumps(run_metadata, indent=2))
print('Training device:', trainer.args.device)
print('Best checkpoint step:', trainer.state.best_global_step)

### Training history

Development scores below are measured before confidence filtering.

In [ ]:
# Summarize losses and development scores saved during training.

training_log = pd.DataFrame([row for row in trainer.state.log_history if 'loss' in row])
epoch_log = pd.DataFrame([row for row in trainer.state.log_history if 'eval_loss' in row])
display(epoch_log[['epoch', 'step', 'eval_loss', 'eval_entity_f1', 'eval_exact_extraction_rate']])
selected_epoch = float(trainer.state.best_global_step) / (trainer.state.global_step / trainer.state.epoch)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(training_log['epoch'], training_log['loss'], color='#8C98AD', label='Training loss')
axes[0].plot(epoch_log['epoch'], epoch_log['eval_loss'], 'o-', color='#526AA2', label='Development loss')
axes[0].set(title='Training loss', xlabel='Epoch', ylabel='Loss (log scale)', yscale='log')
axes[0].legend(frameon=False)
axes[1].plot(epoch_log['epoch'], epoch_log['eval_entity_f1'] * 100, 'o-', color='#526AA2', label='Entity F1')
axes[1].plot(epoch_log['epoch'], epoch_log['eval_exact_extraction_rate'] * 100, 'o-', color='#B497AA', label='Complete-entry accuracy')
axes[1].axvline(selected_epoch, color='#AC785D', linestyle='--', label='Selected checkpoint')
axes[1].set(title='Development performance', xlabel='Epoch', ylabel='Score (%)', ylim=(0, 100))
axes[1].legend(frameon=False, loc='lower right')
for ax in axes:
    ax.set_xticks([1, 2, 3, 4])
    ax.grid(alpha=0.15)
fig.tight_layout()
show_figure(fig, 'training-history')

## 7. Confidence-threshold selection

Evaluate thresholds **0.00**, **0.50**, and **0.80** on the *development set only*. An extracted entity's confidence is the minimum selected-token confidence across its span. Select the threshold by entity F1, breaking ties with complete-entry accuracy. Apply that selected threshold unchanged to the final benchmark.

In [ ]:
# Run entity extraction and tune confidence threshold using development data only.

def predict(text, tokenizer, model, device='cpu'):
    """Predict entity spans and token-derived confidence for one description."""
    import torch
    encoded = tokenizer(text, return_tensors='pt', return_offsets_mapping=True, truncation=False)
    if encoded['input_ids'].shape[1] > MAX_TOKENS:
        raise ValueError('Please describe one subscription in fewer than 128 tokens.')
    offsets = encoded.pop('offset_mapping')[0].tolist()
    inputs = {key: value.to(device) for key, value in encoded.items()}
    with torch.inference_mode():
        probabilities = model(**inputs).logits[0].softmax(-1)
        scores, labels = probabilities.max(-1)
    return decode_entities(text, offsets, labels.cpu().tolist(), scores.cpu().tolist())


def filter_predictions(predictions, threshold):
    """Retain only predicted entities that pass a confidence threshold."""
    return [[entity for entity in entities if entity['confidence'] >= threshold] for entities in predictions]


inference_device = next(model.parameters()).device
development_predictions = [predict(row['text'], tokenizer, model, inference_device) for row in records['development']]
threshold_results = [
    {'threshold': threshold, 'metrics': score_records(records['development'], filter_predictions(development_predictions, threshold))}
    for threshold in (0.0, 0.5, 0.8)
]
selected = max(threshold_results, key=lambda row: (row['metrics']['entity_f1'], row['metrics']['exact_extraction_rate']))
THRESHOLD = selected['threshold']
display(pd.DataFrame([
    {'Threshold': row['threshold'], 'Entity F1 (%)': row['metrics']['entity_f1'] * 100,
     'Complete-entry accuracy (%)': row['metrics']['exact_extraction_rate'] * 100}
    for row in threshold_results
]).round(2))
selection = {
    'threshold': THRESHOLD, 'development_metrics': selected['metrics'],
    'model_sha256': hashlib.sha256((MODEL_DIR / 'model.safetensors').read_bytes()).hexdigest(),
    'development_sha256': manifest['splits']['development']['sha256'],
    'final_sha256': manifest['splits']['final']['sha256'],
}
(OUTPUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2))
print('Selected threshold:', THRESHOLD)

## 8. Comparison with recorded baselines

For research context, compare this run with **recorded** DistilBERT v1 and **recorded** GLiNER Small v2.1 zero-shot development predictions. Only the new DistilBERT model is fine-tuned in this notebook; the baseline models are **not rerun** here. The comparison is therefore informative, but it is not a newly controlled head-to-head training experiment.

In [ ]:
# Compare this training run with recorded baseline predictions (not newly trained baselines).

baselines = [
    ('DistilBERT v1 (recorded)', 'distilbert-v1-development-predictions.json', (0.0, 0.5, 0.8)),
    ('GLiNER zero-shot (recorded)', 'gliner-small-zero-shot-development-predictions.json', (0.3, 0.5, 0.7)),
]
comparison_rows = []
for name, filename, thresholds in baselines:
    predictions = read_json(INPUT_DIR / 'reference' / filename)
    options = [
        {'threshold': threshold, 'metrics': score_records(records['development'], filter_predictions(predictions, threshold))}
        for threshold in thresholds
    ]
    best = max(options, key=lambda row: (row['metrics']['entity_f1'], row['metrics']['exact_extraction_rate']))
    comparison_rows.append({'Model': name, 'Threshold': best['threshold'], **best['metrics']})
comparison_rows.insert(1, {'Model': 'DistilBERT (this run)', 'Threshold': THRESHOLD, **selected['metrics']})
comparison = pd.DataFrame([
    {'Model': row['Model'], 'Threshold': row['Threshold'],
     'Precision (%)': row['entity_precision'] * 100, 'Recall (%)': row['entity_recall'] * 100,
     'Entity F1 (%)': row['entity_f1'] * 100, 'Complete-entry accuracy (%)': row['exact_extraction_rate'] * 100}
    for row in comparison_rows
]).set_index('Model')
display(comparison.round(2))
comparison.to_csv(OUTPUT_DIR / 'development-comparison.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, ['Entity F1 (%)', 'Complete-entry accuracy (%)'], ['Entity F1', 'Complete-entry accuracy']):
    bars = ax.bar(['DistilBERT v1', 'This run', 'GLiNER zero-shot'], comparison[metric], color=['#8C98AD', '#526AA2', '#B497AA'], width=0.6)
    ax.bar_label(bars, fmt='%.2f%%', padding=4)
    ax.set(title=title, ylabel='Score (%)', ylim=(0, 110))
    ax.grid(axis='y', alpha=0.15)
fig.tight_layout()
show_figure(fig, 'model-comparison')

## 9. Evaluation on the final benchmark

Evaluate the development-selected model and threshold on **84 final descriptions**. Entity precision, recall, and F1 require matching field labels and exact text boundaries. **Complete-entry accuracy** requires every expected entity in an individual description to match exactly.

Compare this run's results against the **recorded original experiment** in separate columns, and examine per-field scores, token-level confusion, and extraction errors. Token accuracy and entity-level F1 answer different questions and should not be interpreted interchangeably.

In [ ]:
# Evaluate the selected model on the final benchmark and save prediction files.

final_raw_predictions = [predict(row['text'], tokenizer, model, inference_device) for row in records['final']]
final_predictions = filter_predictions(final_raw_predictions, THRESHOLD)
metrics = score_records(records['final'], final_predictions)
original_metrics = read_json(INPUT_DIR / 'reference/final-metrics.json')['metrics']
achieved = pd.DataFrame([
    {'Metric': name, 'This run (%)': metrics[key] * 100, 'Original Due v2 (%)': original_metrics[key] * 100}
    for name, key in [
        ('Entity precision', 'entity_precision'), ('Entity recall', 'entity_recall'),
        ('Entity F1', 'entity_f1'), ('Complete-entry accuracy', 'exact_extraction_rate'),
    ]
])
display(achieved.round(2))
achieved.to_csv(OUTPUT_DIR / 'final-metrics.csv', index=False)


def entity_keys(entities):
    """Represent entities as comparable (field, start, end) tuples."""
    return {(entity['label'], entity['start'], entity['end']) for entity in entities}


exact = [entity_keys(row['entities']) == entity_keys(prediction) for row, prediction in zip(records['final'], final_predictions)]
complete_count = sum(exact)
print(f'Exactly correct descriptions: {complete_count}/{len(exact)}')
(OUTPUT_DIR / 'final-metrics.json').write_text(json.dumps({'threshold': THRESHOLD, 'metrics': metrics}, indent=2))
(OUTPUT_DIR / 'final-predictions.json').write_text(json.dumps(final_predictions, indent=2, ensure_ascii=False))
fig, ax = plt.subplots(figsize=(10, 4.3))
bars = ax.bar(achieved['Metric'], achieved['This run (%)'], color=['#8C98AD', '#8C98AD', '#526AA2', '#B497AA'], width=0.6)
ax.bar_label(bars, fmt='%.2f%%', padding=5)
ax.set(title='Final results · this run', ylabel='Score (%)', ylim=(0, 111))
ax.grid(axis='y', alpha=0.15)
show_figure(fig, 'final-results')

### Results by field

In [ ]:
# Compute precision, recall, and F1 separately for each subscription field.

field_rows = []
for field, counts in metrics['by_field_counts'].items():
    tp, fp, fn = (counts[key] for key in ('tp', 'fp', 'fn'))
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    field_rows.append({
        'Field': FIELD_NAMES[field], 'Support': tp + fn, 'Correct': tp, 'Extra / wrong': fp, 'Missed': fn,
        'Precision (%)': precision * 100, 'Recall (%)': recall * 100, 'F1 (%)': f1 * 100,
    })
by_field = pd.DataFrame(field_rows).set_index('Field')
display(by_field.round(2))
by_field.to_csv(OUTPUT_DIR / 'per-field-metrics.csv')
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(by_field.index, by_field['F1 (%)'], color='#526AA2', height=0.6)
ax.bar_label(bars, fmt='%.2f%%', padding=4)
ax.set(title='Entity F1 by field · this run', xlabel='F1 (%)', xlim=(0, 112))
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.15)
show_figure(fig, 'per-field-f1')

### Token confusion matrix

Compare raw BIO token predictions with expected labels. Special tokens and padding are excluded. These token scores are separate from the confidence-filtered entity scores.

In [ ]:
# Build a confusion matrix for raw BIO token predictions.

confusion = np.zeros((len(LABELS), len(LABELS)), dtype=int)
for row in records['final']:
    encoded = tokenizer(row['text'], return_tensors='pt')
    with torch.inference_mode():
        tags = model(**{key: value.to(inference_device) for key, value in encoded.items()}).logits[0].argmax(-1).cpu().tolist()
    expected = encode_record(row, tokenizer)['labels']
    for gold, predicted in zip(expected, tags):
        if gold != -100:
            confusion[gold, predicted] += 1
support = confusion.sum(axis=1)
normalized = np.divide(confusion, support[:, None], out=np.zeros_like(confusion, dtype=float), where=support[:, None] != 0)
print(f'Raw BIO token accuracy: {np.trace(confusion) / confusion.sum():.2%}')
axis_labels = ['O'] + [f"{label[0]} · {FIELD_NAMES[label[2:]]}" for label in LABELS[1:]]
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS)), axis_labels, rotation=60, ha='right')
ax.set_yticks(range(len(LABELS)), axis_labels)
ax.set(title='BIO token confusion · this run', xlabel='Predicted label', ylabel='Expected label')
for row in range(len(LABELS)):
    for column in range(len(LABELS)):
        if confusion[row, column]:
            ax.text(column, row, str(confusion[row, column]), ha='center', va='center', fontsize=8,
                    color='white' if normalized[row, column] > 0.5 else '#333945')
fig.colorbar(image, ax=ax, shrink=0.7, label='Row-normalized proportion')
fig.tight_layout()
show_figure(fig, 'token-confusion-matrix')
(OUTPUT_DIR / 'token-confusion.json').write_text(json.dumps({'labels': LABELS, 'counts': confusion.tolist()}, indent=2))

### Extraction errors

In [ ]:
# Collect mistakes to understand missed and extra entity predictions.

errors = []
for row, prediction, correct in zip(records['final'], final_predictions, exact):
    if correct:
        continue
    expected_keys, predicted_keys = entity_keys(row['entities']), entity_keys(prediction)
    missed = [f"{FIELD_NAMES[entity['label']]}: {entity['text']}" for entity in row['entities'] if (entity['label'], entity['start'], entity['end']) not in predicted_keys]
    extra = [f"{FIELD_NAMES[entity['label']]}: {entity['text']}" for entity in prediction if (entity['label'], entity['start'], entity['end']) not in expected_keys]
    errors.append({'Description': row['text'], 'Missing / wrong boundary': '; '.join(missed), 'Extra / wrong boundary': '; '.join(extra)})
print('Descriptions with an extraction error:', len(errors))
display(pd.DataFrame(errors, columns=['Description', 'Missing / wrong boundary', 'Extra / wrong boundary']))
(OUTPUT_DIR / 'final-errors.json').write_text(json.dumps(errors, indent=2, ensure_ascii=False))

## 10. Example inference

The following example uses the model trained **in this notebook**, not a saved output from a previous run. It shows candidate extracted fields and their confidence scores for a new subscription sentence. Predictions are model outputs and can contain mistakes.

In [ ]:
# Demonstrate extraction on a new subscription description.

example_text = 'Netflix costs 15.99 USD per month, renews October 25, and remind me two days before.'
example_entities = [entity for entity in predict(example_text, tokenizer, model, inference_device) if entity['confidence'] >= THRESHOLD]
print(example_text)
display(pd.DataFrame([
    {'Field': FIELD_NAMES[entity['label']], 'Suggested text': entity['text'], 'Span confidence': round(entity['confidence'], 4)}
    for entity in example_entities
], columns=['Field', 'Suggested text', 'Span confidence']))
(MODEL_DIR / 'entry-config.json').write_text(json.dumps({
    'confidence_threshold': THRESHOLD, 'max_tokens': MAX_TOKENS,
    'label_order': LABELS, 'base_model': MODEL_ID, 'base_revision': MODEL_REVISION,
}, indent=2))

## Conclusion and discussion

This notebook presents a named-entity recognition approach for converting natural-language subscription descriptions into structured spans. It reconstructs annotations, checks split integrity, fine-tunes DistilBERT, selects a checkpoint and threshold using development data, and evaluates the resulting model on an existing 84-description benchmark.

The **original** Due v2 experiment reported **97.34% entity F1** and **85.71% complete-entry accuracy** on this benchmark. **This notebook's newly computed results appear in the output below** and may differ from the original experiment. Recorded baselines are provided for context only.

**Limitations and future work:** Training descriptions are synthetic; the authored evaluation set is relatively small and has been used previously; unfamiliar service names, unusual phrasing, and ambiguous dates may be harder. Future research should evaluate a larger, independently collected and carefully de-identified test set, examine error patterns and subgroup coverage, and measure on-device latency and memory before making deployment-performance claims.

In [ ]:
# Display final run metrics and the Kaggle output directory.

display(Markdown(
    f"**This run:** {metrics['entity_f1']:.2%} entity F1 and "
    f"{metrics['exact_extraction_rate']:.2%} complete-entry accuracy "
    f"({complete_count}/{len(records['final'])} descriptions)."
))
print('Model and results saved to:', OUTPUT_DIR)

## References

1. **Pretrained model:** Hugging Face, [DistilBERT base cased](https://huggingface.co/distilbert/distilbert-base-cased). Checkpoint used: `distilbert/distilbert-base-cased` (the notebook pins its revision).
2. **Research paper:** Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). [DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter](https://arxiv.org/abs/1910.01108).
3. **Training framework:** Hugging Face, [Transformers documentation](https://huggingface.co/docs/transformers/index) and [Trainer API](https://huggingface.co/docs/transformers/main_classes/trainer).

**Dataset provenance:** The attached Due Subscription Information Extraction dataset includes its generator patterns, annotated descriptions, split manifest, and original experiment reference artifacts.

## Author: Dol Raj Bashyal